In [1]:
import os
import json
import pandas as pd

from pathlib import Path


# Get GEO_ID from folder name (same pattern as prepare_data.ipynb)
GEO_ID = os.path.basename(os.path.dirname(os.path.abspath('__file__')))


description_path = Path('description.json')
preprocessed_csv_path = Path(f"{GEO_ID}_preprocessed.csv")

if not description_path.exists():
    raise FileNotFoundError(
        "description.json not found. Please run 'prepare_data.ipynb' first to generate it."
    )

if not preprocessed_csv_path.exists():
    raise FileNotFoundError(
        f"{preprocessed_csv_path} not found. Please run 'preprocess_data.ipynb' first to generate it."
    )

with open(description_path) as f:
    summary_data = json.load(f)

df = pd.read_csv(preprocessed_csv_path)

print("Data contains", df.shape[0], "samples and", df.shape[1], "columns.")
print("First few rows of the data:")
df.head()


Data contains 3273 samples and 11582 columns.
First few rows of the data:


,overall survival days,overall survival event,samples,scan-b external id,instrument model,age at diagnosis,tumor size,lymph node group,lymph node status,er status,...,ZSWIM8,ZW10,ZWILCH,ZWINT,ZXDB,ZXDC,ZYG11B,ZYX,ZZEF1,ZZZ3
0,2367,0,F1,Q008818.C008840.S000215.l.r.m2.c.lib.g.k.a.t,HiSeq 2000,43,9.0,NodeNegative,NodeNegative,NaN,...,3.025197,2.918632,1.108204,3.044573,2.425306,3.244432,1.468899,6.281767,1.774107,2.437227
1,2367,0,F2,Q008769.C008792.S000250.l.r.m.c.lib.g.k.a.t,HiSeq 2000,48,14.0,1to3,NodePositive,1.0,...,3.569802,2.551831,1.678388,2.743902,1.861781,2.727427,1.924761,7.169613,1.764214,2.876100
2,2168,1,F3,Q008568.C008577.S000424.l.r.m3.c.lib.g.k.a.t,HiSeq 2000,69,27.0,4toX,NodePositive,1.0,...,5.362272,3.319094,2.443392,4.719843,1.365396,3.122333,1.619277,6.683400,2.039246,2.929346
3,2416,0,F4,Q008909.C009000.S000084.l.r.m.c.lib.g.k.a.t,HiSeq 2000,39,51.0,1to3,NodePositive,1.0,...,4.326709,2.588427,2.441704,4.157466,2.038984,3.182823,1.969433,6.666306,2.771650,2.777674
4,2389,0,F5,Q008781.C008782.S000260.l.r.m.c.lib.g.k.a.t,HiSeq 2000,73,60.0,4toX,NodePositive,1.0,...,4.573915,2.243450,1.749230,1.024820,2.032704,3.588842,2.998474,6.479006,2.696269,3.957146


In [2]:
n_stats_written = 0

for idx, endpoint in enumerate(summary_data['survival-endpoints']):
    time_var = endpoint['time_var']['var_name']
    event_var = endpoint['event_var']['var_name']

    if time_var not in df.columns or event_var not in df.columns:
        abbrv = endpoint.get('abbrv', f'endpoint[{idx}]')
        print(
            f"Skipping stats for {abbrv}: columns not in preprocessed table "
            f"({time_var!r}, {event_var!r})."
        )
        endpoint.pop('stats', None)
        continue

    complete_mask = df[[time_var, event_var]].notnull().all(axis=1)
    n_complete = int(complete_mask.sum())
    n_incomplete = int(len(df) - n_complete)

    # Among samples with complete time + event data
    censored_count = int((complete_mask & (df[event_var] == 0.0)).sum())
    event_count = int((complete_mask & (df[event_var] == 1.0)).sum())

    censored_ratio = censored_count / n_complete

    print(f"Number of samples with incomplete data for {time_var} or {event_var}: {n_incomplete}")
    print(f"Number of samples with complete data for both variables: {n_complete}")
    print(f"Number of censored samples (event_var == 0.0): {censored_count}")
    print(f"Number of events (event_var == 1.0): {event_count}")
    pct = 100.0 * censored_ratio
    print(
        f"Censored ratio (among complete): {censored_count} / {n_complete} = "
        f"{censored_ratio:.3f} ({pct:.1f}%)"
    )

    # Refresh only this block on each run; leave all other endpoint fields unchanged
    endpoint['stats'] = {
        'n_incomplete': n_incomplete,
        'n_complete': n_complete,
        'n_censored': censored_count,
        'n_events': event_count,
        'censored_ratio': round(censored_ratio, 2),
    }
    n_stats_written += 1

with open(description_path, 'w', encoding='utf-8') as f:
    json.dump(summary_data, f, indent=4, ensure_ascii=False)
    f.write('\n')

print(
    f"Updated stats for {n_stats_written} of {len(summary_data['survival-endpoints'])} endpoint(s); "
    f"wrote {description_path.resolve()}"
)


Number of samples with incomplete data for overall survival days or overall survival event: 0
Number of samples with complete data for both variables: 3273
Number of censored samples (event_var == 0.0): 2937
Number of events (event_var == 1.0): 336
Censored ratio (among complete): 2937 / 3273 = 0.897 (89.7%)
Updated stats for 1 of 1 endpoint(s); wrote /Users/jakakokosar/dev/phd/survival-rnaseq-data/datasets/geo/GSE96058/description.json
